#  DecodeLabs Data Science Internship
**Dataset:** Titanic — Passenger Survival Data  
**Tasks covered:** Task 1 → Task 5

---

##  Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

: 

---
##  Task 1 — Data Collection & Dataset Understanding


In [ ]:
# Load the Titanic dataset
df = sns.load_dataset('titanic')

print('=== Dataset Loaded Successfully ===')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns\n')

In [ ]:
# Preview first 5 rows
print('=== First 5 Rows ===')
df.head()

In [ ]:
# Column names and data types
print('=== Columns & Data Types ===')
print(df.dtypes)
print(f'\nTotal features: {df.shape[1]}')

In [ ]:
# Dataset summary info
print('=== Dataset Info ===')
df.info()

In [ ]:
# Describe what each column represents
column_descriptions = {
    'survived'  : 'Target — whether the passenger survived (1) or not (0)',
    'pclass'    : 'Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd)',
    'sex'       : 'Gender of the passenger',
    'age'       : 'Age in years',
    'sibsp'     : 'Number of siblings / spouses aboard',
    'parch'     : 'Number of parents / children aboard',
    'fare'      : 'Passenger fare paid',
    'embarked'  : 'Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)',
    'class'     : 'Ticket class as text',
    'who'       : 'man / woman / child',
    'adult_male': 'Boolean — adult male or not',
    'deck'      : 'Cabin deck letter',
    'embark_town': 'Port name',
    'alive'     : 'yes / no (same as survived)',
    'alone'     : 'Boolean — traveling alone or not'
}

print('=== Column Descriptions ===')
for col, desc in column_descriptions.items():
    print(f'  {col:<15} → {desc}')

---
##  Task 2 — Data Cleaning & Preprocessing


In [ ]:
# Check missing values
print('=== Missing Values Before Cleaning ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f'=== Duplicate Rows ===' )
print(f'Number of duplicates: {duplicates}')

In [ ]:
# Handle missing values
df_clean = df.copy()

# 1. Fill 'age' with median (more robust to outliers than mean)
df_clean['age'].fillna(df_clean['age'].median(), inplace=True)

# 2. Fill 'embarked' with mode (most frequent value)
df_clean['embarked'].fillna(df_clean['embarked'].mode()[0], inplace=True)
df_clean['embark_town'].fillna(df_clean['embark_town'].mode()[0], inplace=True)

# 3. Drop 'deck' — over 77% missing, not reliable
df_clean.drop(columns=['deck'], inplace=True)

print('=== Missing Values After Cleaning ===')
remaining = df_clean.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else 'No missing values remaining!')

In [ ]:
# Format data: encode categorical columns
df_clean['sex_encoded'] = df_clean['sex'].map({'male': 0, 'female': 1})
df_clean['embarked_encoded'] = df_clean['embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Verify data types
print('=== Key Columns After Formatting ===')
print(df_clean[['age', 'sex', 'sex_encoded', 'embarked', 'embarked_encoded']].head(5))

In [ ]:
print(f'=== Dataset Shape ===')
print(f'Before cleaning : {df.shape}')
print(f'After cleaning  : {df_clean.shape}')

---
##  Task 3 — Exploratory Data Analysis (EDA)


In [ ]:
# Basic descriptive statistics
print('=== Descriptive Statistics (Numeric Columns) ===')
df_clean[['age', 'fare', 'sibsp', 'parch']].describe().round(2)

In [ ]:
# Survival rate overall
survival_rate = df_clean['survived'].mean() * 100
print(f'Overall survival rate: {survival_rate:.1f}%')
print(f'Survived    : {df_clean["survived"].sum()}')
print(f'Did not survive: {(df_clean["survived"] == 0).sum()}')

In [ ]:
# Survival by gender
print('=== Survival Rate by Gender ===')
print(df_clean.groupby('sex')['survived'].mean().mul(100).round(1).astype(str) + '%')

In [ ]:
# Survival by class
print('=== Survival Rate by Passenger Class ===')
print(df_clean.groupby('pclass')['survived'].mean().mul(100).round(1).astype(str) + '%')

In [ ]:
# Age distribution stats
print('=== Age Statistics ===')
print(f'Mean age   : {df_clean["age"].mean():.1f}')
print(f'Median age : {df_clean["age"].median():.1f}')
print(f'Std dev    : {df_clean["age"].std():.1f}')
print(f'Min age    : {df_clean["age"].min()}')
print(f'Max age    : {df_clean["age"].max()}')

In [ ]:
# Outlier detection in 'fare' using IQR
Q1 = df_clean['fare'].quantile(0.25)
Q3 = df_clean['fare'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_clean[(df_clean['fare'] < Q1 - 1.5*IQR) | (df_clean['fare'] > Q3 + 1.5*IQR)]

print(f'=== Fare Outliers (IQR Method) ===')
print(f'Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}')
print(f'Outlier threshold: > {Q3 + 1.5*IQR:.2f}')
print(f'Number of fare outliers: {len(outliers)}')

In [ ]:
# Correlation matrix
print('=== Correlation with Survived ===')
corr = df_clean[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_encoded']].corr()
print(corr['survived'].sort_values(ascending=False).round(3))

---
##  Task 4 — Data Visualization


In [ ]:
# Set style
sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Titanic Dataset — Exploratory Visualizations', fontsize=16, fontweight='bold', y=1.01)

# 1. Survival count
sns.countplot(data=df_clean, x='survived', ax=axes[0, 0],
              palette=['#e07070', '#70b070'])
axes[0, 0].set_title('Survival Count')
axes[0, 0].set_xticklabels(['Did Not Survive', 'Survived'])
axes[0, 0].set_xlabel('')

# 2. Survival by gender
sns.barplot(data=df_clean, x='sex', y='survived', ax=axes[0, 1],
            palette='Set2', ci=None)
axes[0, 1].set_title('Survival Rate by Gender')
axes[0, 1].set_ylabel('Survival Rate')
axes[0, 1].set_ylim(0, 1)

# 3. Survival by class
sns.barplot(data=df_clean, x='pclass', y='survived', ax=axes[0, 2],
            palette='Blues_d', ci=None)
axes[0, 2].set_title('Survival Rate by Passenger Class')
axes[0, 2].set_ylabel('Survival Rate')
axes[0, 2].set_ylim(0, 1)

# 4. Age distribution by survival
df_clean[df_clean['survived'] == 0]['age'].plot.kde(ax=axes[1, 0], label='Did Not Survive', color='#e07070')
df_clean[df_clean['survived'] == 1]['age'].plot.kde(ax=axes[1, 0], label='Survived', color='#70b070')
axes[1, 0].set_title('Age Distribution by Survival')
axes[1, 0].set_xlabel('Age')
axes[1, 0].legend()

# 5. Fare boxplot by class
sns.boxplot(data=df_clean, x='pclass', y='fare', ax=axes[1, 1],
            palette='Set3')
axes[1, 1].set_title('Fare Distribution by Class')
axes[1, 1].set_xlabel('Passenger Class')

# 6. Heatmap — correlation
corr_data = df_clean[['survived', 'pclass', 'age', 'fare', 'sibsp', 'parch', 'sex_encoded']].corr()
sns.heatmap(corr_data, annot=True, fmt='.2f', ax=axes[1, 2],
            cmap='RdYlGn', linewidths=0.5, vmin=-1, vmax=1)
axes[1, 2].set_title('Correlation Heatmap')

plt.tight_layout()
plt.savefig('titanic_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualizations saved as titanic_visualizations.png')

In [ ]:
# Extra: Survival by gender AND class (grouped)
plt.figure(figsize=(8, 5))
sns.barplot(data=df_clean, x='pclass', y='survived', hue='sex',
            palette='Set1', ci=None)
plt.title('Survival Rate by Class and Gender')
plt.ylabel('Survival Rate')
plt.xlabel('Passenger Class')
plt.ylim(0, 1)
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

---
##  Task 5 — Predictive Model (Logistic Regression)


In [ ]:
# Select features for the model
features = ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_encoded', 'embarked_encoded']
target   = 'survived'

X = df_clean[features]
y = df_clean[target]

print(f'Features used: {features}')
print(f'Target       : {target}')
print(f'Dataset size : {X.shape[0]} samples, {X.shape[1]} features')

In [ ]:
# Train / Test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')

In [ ]:
# Build and train Logistic Regression model
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)

print('Model trained successfully!')

In [ ]:
# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'=== Model Accuracy ===')
print(f'Accuracy: {accuracy * 100:.2f}%\n')

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

In [ ]:
# Confusion matrix visualization
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: No', 'Predicted: Yes'],
            yticklabels=['Actual: No', 'Actual: Yes'])
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (model coefficients)
coeff_df = pd.DataFrame({
    'Feature'    : features,
    'Coefficient': model.coef_[0].round(3)
}).sort_values('Coefficient', ascending=False)

print('=== Feature Coefficients (Impact on Survival) ===')
print(coeff_df.to_string(index=False))

plt.figure(figsize=(8, 5))
colors = ['#70b070' if c > 0 else '#e07070' for c in coeff_df['Coefficient']]
plt.barh(coeff_df['Feature'], coeff_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Coefficients — Effect on Survival Probability')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
# Predict survival for a new passenger
print('=== Predict a New Passenger ===')

# Example: 25-year-old female, 1st class, no siblings, paid 100
new_passenger = pd.DataFrame([{
    'pclass'           : 1,
    'age'              : 25,
    'sibsp'            : 0,
    'parch'            : 0,
    'fare'             : 100,
    'sex_encoded'      : 1,   # female
    'embarked_encoded' : 0    # Southampton
}])

pred = model.predict(new_passenger)[0]
prob = model.predict_proba(new_passenger)[0]

print(f'Passenger profile: 25-year-old female, 1st class, fare=100')
print(f'Prediction        : {"SURVIVED" if pred == 1 else "DID NOT SURVIVE"}')
print(f'Survival probability : {prob[1]*100:.1f}%')